In [1]:
import pandas as pd, numpy as np
df = pd.read_csv("sdss_100k_galaxy_form_burst.csv", comment='#')
print(df.shape)
print(df['subclass'].value_counts())

(4857, 43)
subclass
STARFORMING    3627
STARBURST      1229
Name: count, dtype: int64


In [2]:
import os
path = "../sdss_100k_galaxy_form_burst.csv"
print("Looking at:", os.path.abspath(path))
print("Size (MB):", os.path.getsize(path) / 1e6)

with open(path, encoding="utf-8", errors="ignore") as f:
    print("Lines in file:", sum(1 for _ in f))

df2 = pd.read_csv(path)   # same file, but without comment='#'
print("Rows without comment option:", df2.shape)

Looking at: /sdss_100k_galaxy_form_burst.csv


FileNotFoundError: [Errno 2] No such file or directory: '../sdss_100k_galaxy_form_burst.csv'

In [3]:
import os, pandas as pd
print(os.listdir("."))              # see which files are here

path = "sdss_100k_galaxy_form_burst.csv"
print("Size (MB):", os.path.getsize(path) / 1e6)

with open(path, encoding="utf-8", errors="ignore") as f:
    print("Lines in file:", sum(1 for _ in f))

df2 = pd.read_csv(path)             # without comment='#'
print("Rows without comment option:", df2.shape)

['.config', 'sdss_100k_galaxy_form_burst.csv', 'sample_data']
Size (MB): 43.20791
Lines in file: 100002


/tmp/ipykernel_1172/593652320.py:10: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,41,42) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv(path)             # without comment='#'


Rows without comment option: (100001, 1)


In [4]:
with open("sdss_100k_galaxy_form_burst.csv", encoding="utf-8", errors="ignore") as f:
    for _ in range(3):
        print(repr(f.readline()[:150]))

df = pd.read_csv("sdss_100k_galaxy_form_burst.csv", skiprows=1)
print(df.shape)
print(df['subclass'].value_counts())

'#Table1\n'
'objid,specobjid,ra,dec,u,g,r,i,z,modelFlux_u,modelFlux_g,modelFlux_r,modelFlux_i,modelFlux_z,petroRad_u,petroRad_g,petroRad_i,petroRad_r,petroRad_z,pe'
'1237646587710669400,8175185722644649984,82.0386790197966,0.847177136346427,21.73818,20.26633,19.32409,18.64037,18.23833,2.007378,7.82364,18.63581,34.9'
(100000, 43)
subclass
STARFORMING    74993
STARBURST      25007
Name: count, dtype: int64


In [5]:
# 1. Turn -9999 into "missing" (NaN). SDSS uses -9999 to mean "not measured".
df = df.replace(-9999, np.nan)
print(df.isna().sum()[df.isna().sum() > 0])   # columns that have missing values

# 2. Drop columns that are not real measurements
#    objid, specobjid = ID numbers; ra, dec = sky position; class = "GALAXY" for every row
df = df.drop(columns=["objid", "specobjid", "ra", "dec", "class"])

# 3. Turn the label into numbers: STARFORMING = 0, STARBURST = 1
df["subclass"] = df["subclass"].map({"STARFORMING": 0, "STARBURST": 1})
print(df["subclass"].value_counts())

# 4. Separate inputs (X) and the label (y)
X = df.drop(columns=["subclass"])
y = df["subclass"]

# 5. Split BEFORE anything else, keeping the 75/25 ratio in both parts
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

u               11
g               11
r                9
i               10
z               13
petroR50_u    1570
petroR50_g     154
petroR50_i     131
petroR50_r      68
petroR50_z     923
psfMag_u        11
psfMag_r         9
psfMag_g        11
psfMag_i        10
psfMag_z        13
expAB_u         11
expAB_g         11
expAB_r          9
expAB_i         10
expAB_z         13
dtype: int64
subclass
0    74993
1    25007
Name: count, dtype: int64
(80000, 37) (20000, 37)
0.250075 0.25005


In [7]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

models = {
    "Always guess majority": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                         LogisticRegression(max_iter=1000)),
    "Random Forest": make_pipeline(SimpleImputer(strategy="median"),
                                   RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Macro F1": f1_score(y_test, pred, average="macro"),
        "Starburst precision": precision_score(y_test, pred, zero_division=0),
        "Starburst recall": recall_score(y_test, pred),
        "Starburst F1": f1_score(y_test, pred),
    })

results = pd.DataFrame(rows).round(3)
print(results)

                   Model  Accuracy  Macro F1  Starburst precision  \
0  Always guess majority     0.750     0.429                0.000   
1    Logistic Regression     0.867     0.808                0.795   
2          Random Forest     0.893     0.852                0.822   

   Starburst recall  Starburst F1  
0             0.000         0.000  
1             0.629         0.702  
2             0.732         0.775  


In [8]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

def make_rf(**kw):
    return RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, **kw)

setups = {
    "RF: no handling": ImbPipeline([("imp", SimpleImputer(strategy="median")),
                                    ("clf", make_rf())]),
    "RF: class weights": ImbPipeline([("imp", SimpleImputer(strategy="median")),
                                      ("clf", make_rf(class_weight="balanced"))]),
    "RF: SMOTE": ImbPipeline([("imp", SimpleImputer(strategy="median")),
                              ("smote", SMOTE(random_state=42)),
                              ("clf", make_rf())]),
}

rows = []
for name, model in setups.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rows.append({
        "Setup": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Macro F1": f1_score(y_test, pred, average="macro"),
        "Starburst precision": precision_score(y_test, pred),
        "Starburst recall": recall_score(y_test, pred),
        "Starburst F1": f1_score(y_test, pred),
    })

print(pd.DataFrame(rows).round(3))

               Setup  Accuracy  Macro F1  Starburst precision  \
0    RF: no handling     0.893     0.852                0.822   
1  RF: class weights     0.890     0.845                0.834   
2          RF: SMOTE     0.882     0.849                0.732   

   Starburst recall  Starburst F1  
0             0.732         0.775  
1             0.701         0.762  
2             0.830         0.778  


In [9]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.ensemble import HistGradientBoostingClassifier
from xgboost import XGBClassifier

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"macro_f1": "f1_macro", "sb_precision": "precision",
           "sb_recall": "recall", "sb_f1": "f1"}

candidates = {
    "Logistic Regression": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                         LogisticRegression(max_iter=1000)),
    "Random Forest": make_pipeline(SimpleImputer(strategy="median"),
                                   RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
    "Hist Gradient Boosting": HistGradientBoostingClassifier(random_state=42),
    "XGBoost": XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6,
                             eval_metric="logloss", random_state=42, n_jobs=-1),
}

rows = []
for name, model in candidates.items():
    res = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
    row = {"Model": name}
    for key in scoring:
        scores = res[f"test_{key}"]
        row[key] = f"{scores.mean():.3f} ± {scores.std():.3f}"
    rows.append(row)
    print("done:", name)

print(pd.DataFrame(rows))

done: Logistic Regression
done: Random Forest
done: Hist Gradient Boosting
done: XGBoost
                    Model       macro_f1   sb_precision      sb_recall  \
0     Logistic Regression  0.816 ± 0.009  0.805 ± 0.009  0.643 ± 0.019   
1           Random Forest  0.852 ± 0.002  0.820 ± 0.003  0.734 ± 0.005   
2  Hist Gradient Boosting  0.862 ± 0.002  0.820 ± 0.005  0.762 ± 0.005   
3                 XGBoost  0.864 ± 0.003  0.821 ± 0.004  0.768 ± 0.007   

           sb_f1  
0  0.715 ± 0.015  
1  0.774 ± 0.003  
2  0.790 ± 0.003  
3  0.794 ± 0.004  


In [10]:
# ---------- Part A: final test score (use the test set once) ----------
final = XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6,
                      eval_metric="logloss", random_state=42, n_jobs=-1)
final.fit(X_train, y_train)
pred = final.predict(X_test)
print("TEST  accuracy:", round(accuracy_score(y_test, pred), 3),
      "| macro F1:", round(f1_score(y_test, pred, average="macro"), 3),
      "| starburst precision:", round(precision_score(y_test, pred), 3),
      "| recall:", round(recall_score(y_test, pred), 3),
      "| F1:", round(f1_score(y_test, pred), 3))

# ---------- Part B: ablation (cross-validation on training data only) ----------
groups = {
    "redshift": ["redshift"],
    "redshift_err": ["redshift_err"],
    "magnitudes (u,g,r,i,z)": ["u", "g", "r", "i", "z"],
    "psfMag": [c for c in X.columns if c.startswith("psfMag")],
    "modelFlux": [c for c in X.columns if c.startswith("modelFlux")],
    "petroFlux": [c for c in X.columns if c.startswith("petroFlux")],
    "size (petroRad, petroR50)": [c for c in X.columns if c.startswith(("petroRad", "petroR50"))],
    "expAB (shape)": [c for c in X.columns if c.startswith("expAB")],
}

def cv_score(cols):
    m = XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6,
                      eval_metric="logloss", random_state=42, n_jobs=-1)
    res = cross_validate(m, X_train[cols], y_train, cv=cv,
                         scoring={"f1": "f1", "rec": "recall"})
    return res["test_f1"].mean(), res["test_rec"].mean()

all_cols = list(X_train.columns)
base_f1, base_rec = cv_score(all_cols)
rows = [{"Removed": "nothing (all features)", "Starburst F1": round(base_f1, 3),
         "Starburst recall": round(base_rec, 3), "F1 change": 0.0}]
for name, cols in groups.items():
    keep = [c for c in all_cols if c not in cols]
    f1v, recv = cv_score(keep)
    rows.append({"Removed": name, "Starburst F1": round(f1v, 3),
                 "Starburst recall": round(recv, 3), "F1 change": round(f1v - base_f1, 3)})
    print("done:", name)

print(pd.DataFrame(rows))

# ---------- Part C: which features the model leans on most ----------
imp = pd.Series(final.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(imp.head(10).round(3))

TEST  accuracy: 0.902 | macro F1: 0.866 | starburst precision: 0.825 | recall: 0.77 | F1: 0.797
done: redshift
done: redshift_err
done: magnitudes (u,g,r,i,z)
done: psfMag
done: modelFlux
done: petroFlux
done: size (petroRad, petroR50)
done: expAB (shape)
                     Removed  Starburst F1  Starburst recall  F1 change
0     nothing (all features)         0.794             0.768      0.000
1                   redshift         0.774             0.748     -0.020
2               redshift_err         0.757             0.724     -0.036
3     magnitudes (u,g,r,i,z)         0.794             0.767      0.001
4                     psfMag         0.788             0.758     -0.006
5                  modelFlux         0.793             0.768     -0.000
6                  petroFlux         0.794             0.769      0.000
7  size (petroRad, petroR50)         0.792             0.766     -0.001
8              expAB (shape)         0.793             0.765     -0.001
petroRad_g      0.177
mo

In [11]:
sets = {
    "All features": all_cols,
    "Without redshift_err": [c for c in all_cols if c != "redshift_err"],
    "Without redshift & redshift_err (photometry only)":
        [c for c in all_cols if c not in ("redshift", "redshift_err")],
}
rows = []
for name, cols in sets.items():
    f1v, recv = cv_score(cols)
    rows.append({"Features": name, "Starburst F1": round(f1v, 3), "Starburst recall": round(recv, 3)})
    print("done:", name)
print(pd.DataFrame(rows))

done: All features
done: Without redshift_err
done: Without redshift & redshift_err (photometry only)
                                            Features  Starburst F1  \
0                                       All features         0.794   
1                               Without redshift_err         0.757   
2  Without redshift & redshift_err (photometry only)         0.749   

   Starburst recall  
0             0.768  
1             0.724  
2             0.718  


In [12]:
compact = ["u", "g", "r", "i", "z",
           "petroR50_r", "petroR50_g", "petroRad_r", "petroRad_g", "expAB_r"]
photo_full = [c for c in X_train.columns if c not in ("redshift", "redshift_err")]

for name, cols in {"Photometry-only, all 35": photo_full,
                   "Compact, 10 features": compact}.items():
    f1v, recv = cv_score(cols)
    print(name, "| starburst F1:", round(f1v, 3), "| recall:", round(recv, 3))

Photometry-only, all 35 | starburst F1: 0.749 | recall: 0.718
Compact, 10 features | starburst F1: 0.744 | recall: 0.71


In [13]:
import joblib, json
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

final_cols = compact

model = XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6,
                      eval_metric="logloss", random_state=42, n_jobs=-1)
model.fit(X_train[final_cols], y_train)

pred = model.predict(X_test[final_cols])
print(classification_report(y_test, pred, digits=3))

joblib.dump(model, "galaxy_model.joblib")
json.dump(final_cols, open("model_features.json", "w"))

from google.colab import files
files.download("galaxy_model.joblib")
files.download("model_features.json")

              precision    recall  f1-score   support

           0      0.904     0.931     0.918     14999
           1      0.773     0.704     0.737      5001

    accuracy                          0.875     20000
   macro avg      0.839     0.818     0.827     20000
weighted avg      0.872     0.875     0.872     20000



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>